In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot  as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

In [13]:
def pre_process(filename):
    # Load data
    df = pd.read_csv(filename, index_col='id')
    df['date_sold'] = pd.to_datetime(df['date_sold'])
    df['year_sold'] = df['date_sold'].dt.year

    # Drop irrelevant columns
    df.drop([
        'suburb_lat','date_sold','ethnic_breakdown','suburb_elevation',
        'cash_rate','suburb_lng','suburb_sqkm','suburb_population','median_house_rent_per_week',
        'median_apartment_rent_per_week','suburbpopulation','public_housing_pct','postcode',
        'nearest_train_station','highlights_attractions', 'ideal_for', 'traffic', 'public_transport', 'affordability_rental',
        'affordability_buying', 'nature', 'noise', 'things_to_see_do', 'family_friendliness', 'pet_friendliness', 'safety', 'overall_rating'
    ], axis=1, inplace=True)

    # String columns
    df['suburb'] = df['suburb'].astype("string")
    df['type'] = df['type'].astype("string")
    df['region'] = df['region'].astype("string")
    df['time_to_cbd_public_transport_town_hall_st'] = df['time_to_cbd_public_transport_town_hall_st'].fillna(df['time_to_cbd_public_transport_town_hall_st'].mean())
    df['time_to_cbd_driving_town_hall_st'] = df['time_to_cbd_driving_town_hall_st'].fillna(df['time_to_cbd_driving_town_hall_st'].mean())
    df['commute_time'] = (df['time_to_cbd_public_transport_town_hall_st']+df['time_to_cbd_driving_town_hall_st'])/2
    df=df.drop(['time_to_cbd_public_transport_town_hall_st','time_to_cbd_driving_town_hall_st'],axis=1)

    def choose_median_price(row):
        property_type = row['type'].lower()
        if property_type == 'house':
            return row['suburb_median_house_price']
        elif 'apartment' in property_type:
            return row['suburb_median_apartment_price']
        else:
            return (row['suburb_median_apartment_price'] + row['suburb_median_house_price']) / 2

    df['suburb_median_price'] = df.apply(choose_median_price, axis=1)
    df=df.drop(['suburb_median_apartment_price','suburb_median_house_price'],axis=1)

    # Add new features
    df['num_of_rooms'] = df['num_bed'] + df['num_bath']
    df['is_sold_in_2021'] = (df['year_sold'] == 2021).astype(int)
    df['years_diff'] = 2022 - df['year_sold']
    df['inverse_cbd_distance'] = 1 / (df['km_from_cbd'] + 1)
    df['is_cbd'] = (df['km_from_cbd'] < 10).astype(int)

    # Drop redundant
    df.drop(['year_sold','km_from_cbd'], axis=1, inplace=True)

    # Region 
    region_map = {
        'South West': 0,
        'Western Suburbs': 1,
        'Hills Shire': 2,
        'Inner South': 3,
        'Sutherland Shire': 4,
        'Southern Suburbs': 5,
        'Northern Suburbs': 6,
        'Sydney City': 7,
        'Inner West': 8,
        'Inner East': 9,
        'Northern Beaches': 10,
        'North Shore': 11,
        'Upper North Shore': 12,
        'Lower North Shore': 13,
        'Eastern Suburbs': 14
    }

    df['region_group'] = df['region'].map(region_map)
    df['low_suburb_price'] = df['region_group'].isin([0,1]).astype(int)
    df['high_suburb_price'] = df['region_group'].isin([13,14]).astype(int)    
    return df

In [14]:
train_df = pre_process("train.csv")
test_df = pre_process("test.csv")

In [15]:
def feature_engineering(df):
    #Split the Features
    discrete_threshold = 25
    numerical_discrete = []
    numerical_continuous = []
    categorical_features = []
    exclude_cols = ['id','price']
    for col in df.columns:
        if col not in exclude_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                unique_vals = df[col].nunique()
                if unique_vals <= discrete_threshold :
                    numerical_discrete.append(col)
                else:
                    numerical_continuous.append(col)
            else:
                categorical_features.append(col)
        
    print("Discrete numerical features:")
    print(numerical_discrete)
    print("\nContinuous numerical features:")
    print(numerical_continuous)
    print("\nCategorical features:")
    print(categorical_features )
    
    return numerical_discrete,numerical_continuous,categorical_features

In [16]:
# discrete_features,continuous_features,categorical_features=feature_engineering(train_df)
features = train_df.drop('price' ,axis=1).columns
target = 'price'
print(features)

Discrete numerical features:
['num_bath', 'num_bed', 'num_parking', 'property_inflation_index', 'is_sold_in_2021', 'years_diff', 'is_cbd', 'region_group', 'low_suburb_price', 'high_suburb_price']

Continuous numerical features:
['property_size', 'suburb_median_income', 'avg_years_held', 'commute_time', 'suburb_median_price', 'num_of_rooms', 'inverse_cbd_distance']

Categorical features:
['suburb', 'type', 'region']


In [21]:
def regression_hgb(train_df, test_df, features, target):
    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

    # Train the model
    model = HistGradientBoostingRegressor(
        learning_rate=0.15,
        max_iter=500,
        max_depth=7,
        loss = 'absolute_error',
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict on both train and test sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print(f"\n🛠️ Train MAE: {train_mae:.2f}")
    print(f"🧪 Test MAE: {test_mae:.2f}")

    return train_mae, test_mae


In [22]:
regression_hgb(train_df, test_df, features, target)


🛠️ Train MAE: 196327.22
🧪 Test MAE: 305068.91


(196327.22068263168, 305068.9057316051)